# Phase 9 — Baseline

## Objective

The purpose of this phase is to establish simple and transparent baseline approaches before applying machine learning models.

The baselines will provide reference rankings that can later be compared with more advanced scoring and machine learning approaches.

The baseline approaches should be:

- Simple
- Reproducible
- Interpretable
- Based only on information available in the analytical dataset
- Easy to compare with later models

The main objective is not to create the final opportunity-scoring system at this stage.

Instead, the baseline provides a reference point for determining whether more advanced approaches add useful ranking value.

In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute("PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp/duckdb_tmp';")
con.execute("PRAGMA memory_limit='1GB';")

In [4]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

In [5]:
con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE performance_aggregated AS
    SELECT
        client_hash_id,
        content_hash_id,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS reporting_days,
        SUM(gsc_impressions) AS total_gsc_impressions,
        SUM(gsc_clicks) AS total_gsc_clicks,
        SUM(gsc_sum_position) AS total_gsc_sum_position,
        AVG(gsc_avg_position) AS mean_gsc_avg_position
    FROM performance_deduplicated
    GROUP BY client_hash_id, content_hash_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
con.execute(f"""
    CREATE OR REPLACE TABLE analytical_dataset AS
    SELECT
        p.client_hash_id,
        p.content_hash_id,

        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted,

        p.first_report_date,
        p.last_report_date,
        p.reporting_days,
        p.total_gsc_impressions,
        p.total_gsc_clicks,
        p.total_gsc_sum_position,
        p.mean_gsc_avg_position

    FROM performance_aggregated p
    INNER JOIN read_parquet('{content_file}') c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""")

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE feature_dataset AS
    SELECT
        *,
        
        CASE
            WHEN total_gsc_impressions > 0
            THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            ELSE NULL
        END AS ctr,

        CASE
            WHEN reporting_days > 0
            THEN total_gsc_impressions * 1.0 / reporting_days
            ELSE NULL
        END AS impressions_per_reporting_day,

        CASE
            WHEN reporting_days > 0
            THEN total_gsc_clicks * 1.0 / reporting_days
            ELSE NULL
        END AS clicks_per_reporting_day,

        LN(1 + total_gsc_impressions) AS log_impressions,
        LN(1 + total_gsc_clicks) AS log_clicks,

        CASE
            WHEN backlinks IS NOT NULL
            THEN LN(1 + backlinks)
            ELSE NULL
        END AS log_backlinks,

        CASE
            WHEN search_volume IS NOT NULL
            THEN LN(1 + search_volume)
            ELSE NULL
        END AS log_search_volume

    FROM analytical_dataset
""")

In [9]:
# -------------------------
# ---Impressions Ranking---
# -------------------------

baseline_impressions = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_impressions DESC
        ) AS rank_impressions

    FROM feature_dataset
""").df()

baseline_impressions.head(10)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_impressions
0,client_e547b89c05043229,content_963de14b1f58978f,615012.0,1676.0,6.397556,30,1
1,client_e547b89c05043229,content_eadb33b5df496f4a,591696.0,3817.0,2.258612,30,2
2,client_e547b89c05043229,content_545bb6cc7081ded3,585712.0,3048.0,2.110776,30,3
3,client_8ddc46da5414ffd8,content_943dc881428182b8,292416.0,399.0,2.689206,30,4
4,client_9c26c096d6e57253,content_adcc7b85a04c187d,283505.0,152170.0,1.723897,30,5
5,client_06d356715a8ff3b6,content_f88878f155e4838d,277353.0,2543.0,5.034392,30,6
6,client_e547b89c05043229,content_9ef3d7516483e665,269943.0,787.0,2.098715,30,7
7,client_9c26c096d6e57253,content_54f2b96801c90591,267068.0,142612.0,1.731033,29,8
8,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,235427.0,45.0,7.132457,30,9
9,client_8ddc46da5414ffd8,content_b902320872acab45,234902.0,303.0,5.510291,30,10


In [10]:
# --------------------
# ---Clicks Ranking---
# --------------------

baseline_clicks = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_clicks DESC
        ) AS rank_clicks

    FROM feature_dataset
""").df()

baseline_clicks.head(10)


,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_clicks
0,client_9c26c096d6e57253,content_adcc7b85a04c187d,283505.0,152170.0,1.723897,30,1
1,client_9c26c096d6e57253,content_54f2b96801c90591,267068.0,142612.0,1.731033,29,2
2,client_08a6a72ff48e62c0,content_b40551e4d1418a3d,87710.0,4929.0,2.744005,30,3
3,client_08a6a72ff48e62c0,content_56d2e24db684b94e,87710.0,4929.0,2.744005,30,4
4,client_08a6a72ff48e62c0,content_54d5ce7b65ee6401,87710.0,4929.0,2.744005,30,5
5,client_e547b89c05043229,content_eadb33b5df496f4a,591696.0,3817.0,2.258612,30,6
6,client_e547b89c05043229,content_545bb6cc7081ded3,585712.0,3048.0,2.110776,30,7
7,client_b77d0d5f08f05e64,content_ac1ddc0c0e79289f,181605.0,2863.0,5.471849,30,8
8,client_06d356715a8ff3b6,content_f88878f155e4838d,277353.0,2543.0,5.034392,30,9
9,client_73cda7b4e4f265ea,content_dd27d50fad609dad,43231.0,2203.0,4.237566,29,10


In [11]:
# ----------------------
# ---Average Position---
# --------------------

baseline_position = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY mean_gsc_avg_position ASC NULLS LAST
        ) AS rank_position

    FROM feature_dataset
    WHERE mean_gsc_avg_position IS NOT NULL
""").df()

baseline_position.head(10)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_position
0,client_157ffe4d4a595515,content_f1ffcbd5101a995b,1.0,0.0,0.0,30,1
1,client_157ffe4d4a595515,content_f760df0d7bf9ff9c,1.0,0.0,0.0,30,2
2,client_157ffe4d4a595515,content_fc367a1b617ba8ca,1.0,0.0,0.0,30,3
3,client_157ffe4d4a595515,content_fd120a58bc95554c,1.0,0.0,0.0,30,4
4,client_3f0ce4d44fe94f3d,content_12b9e32db20629f7,1.0,0.0,0.0,30,5
5,client_3f0ce4d44fe94f3d,content_137e5470bda1ec04,1.0,0.0,0.0,30,6
6,client_3f0ce4d44fe94f3d,content_140798ff5e3575af,1.0,0.0,0.0,30,7
7,client_3f0ce4d44fe94f3d,content_14a74d390aa310be,1.0,0.0,0.0,30,8
8,client_3f0ce4d44fe94f3d,content_14a9a3c824c6393a,1.0,0.0,0.0,30,9
9,client_3f0ce4d44fe94f3d,content_14f65b8b5a440380,1.0,0.0,0.0,30,10


In [12]:
# -----------------------------
# ---Simple Opportunity Rule---
# -----------------------------

baseline_opportunity = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        CASE
            WHEN total_gsc_impressions > 0
                 AND mean_gsc_avg_position >= 11
            THEN 1
            ELSE 0
        END AS opportunity_flag

    FROM feature_dataset
""").df()

baseline_opportunity['opportunity_flag'].value_counts()

opportunity_flag
0    295204
1    114001
Name: count, dtype: int64

In [13]:
# -------------------------------
# ---Analyze the Baseline Rule---
# -------------------------------

opportunity_summary = baseline_opportunity['opportunity_flag'].value_counts().to_frame('content_count')

opportunity_summary['percentage'] = (
    opportunity_summary['content_count']
    / len(baseline_opportunity)
    * 100
)

opportunity_summary

,content_count,percentage
opportunity_flag,,
0,295204,72.140858
1,114001,27.859142


In [14]:
baseline_opportunity.groupby('opportunity_flag')[
    [
        'total_gsc_impressions',
        'total_gsc_clicks',
        'mean_gsc_avg_position',
        'reporting_days'
    ]
].agg(['count', 'mean', 'median'])

total_gsc_impressions                    total_gsc_clicks  \
                                 count        mean median            count   
opportunity_flag                                                             
0                               295204  566.045734    0.0           295204   
1                               114001  430.143692   86.0           114001   

                                  mean_gsc_avg_position                        \
                      mean median                 count       mean     median   
opportunity_flag                                                                
0                 3.516152    0.0                 94635   6.216735   6.550112   
1                 1.498627    0.0                114001  36.803116  29.432262   

                 reporting_days                    
                          count       mean median  
opportunity_flag                                   
0                        295204  28.237951   30.0  
1                        114001  29.400847   30.0

In [15]:
con.execute("""
    SELECT
        opportunity_flag,
        COUNT(*) AS content_count,
        AVG(total_gsc_impressions) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,
        AVG(total_gsc_clicks) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,
        AVG(mean_gsc_avg_position) AS avg_position,
        MEDIAN(mean_gsc_avg_position) AS median_position
    FROM (
        SELECT
            *,
            CASE
                WHEN total_gsc_impressions > 0
                     AND mean_gsc_avg_position >= 11
                THEN 1
                ELSE 0
            END AS opportunity_flag
        FROM feature_dataset
    )
    GROUP BY opportunity_flag
    ORDER BY opportunity_flag
""").df()

,opportunity_flag,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_position,median_position
0,0,295204,566.045734,0.0,3.516152,0.0,6.216735,6.550112
1,1,114001,430.143692,86.0,1.498627,0.0,36.803116,29.432262


In [16]:
con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days
    FROM feature_dataset
    WHERE total_gsc_impressions > 0
      AND mean_gsc_avg_position >= 11
    ORDER BY total_gsc_impressions DESC
    LIMIT 20
""").df()

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days
0,client_a80fca3f171ed1de,content_012de75c008aa653,179662.0,0.0,18.276568,30
1,client_23a62021009f63c4,content_c60628276389acbb,123819.0,6.0,73.408960,30
2,client_23a62021009f63c4,content_16b29599a771ed7d,117001.0,191.0,15.938518,30
3,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,107215.0,0.0,21.624960,30
4,client_23a62021009f63c4,content_661a7734f691bef5,106836.0,180.0,23.705431,30
5,client_e547b89c05043229,content_ba98b9ba325cb149,103160.0,52.0,41.306458,30
6,client_23a62021009f63c4,content_68ffc95c19f61219,97740.0,15.0,24.407208,30
7,client_fef1a8f436438636,content_ba462518dad435fc,90539.0,14.0,23.609160,30
8,client_23a62021009f63c4,content_e8a52cf3d5988c07,81577.0,317.0,11.855581,30
9,client_23a62021009f63c4,content_36e53e9c707674fc,80453.0,213.0,27.351133,30


## Task 9.7 — Baseline Comparison

Multiple simple ranking strategies are compared to establish reference approaches before applying machine learning.

The following baselines are considered:

1. Total impressions ranking
2. Total clicks ranking
3. Average position ranking
4. A simple opportunity rule based on search visibility and average position

The purpose is to understand how different simple signals prioritize content items and whether they produce substantially different rankings.

In [17]:
# -------------------------------
# Create a Combined Ranking Table
# -------------------------------

baseline_comparison = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_impressions DESC
        ) AS rank_impressions,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_clicks DESC
        ) AS rank_clicks,

        ROW_NUMBER() OVER (
            ORDER BY mean_gsc_avg_position ASC NULLS LAST
        ) AS rank_position

    FROM feature_dataset
""").df()

baseline_comparison.head(10)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_impressions,rank_clicks,rank_position
0,client_cd12bcfd98942aa1,content_909c1d1f58b0b197,5.0,0.0,0.0,26,172519,384494,1
1,client_f623b01661d4bfe4,content_1945fc2ff78dd7f9,1.0,1.0,0.0,25,207521,72655,2
2,client_f623b01661d4bfe4,content_baf235ef423b1335,1.0,1.0,0.0,25,205042,73003,3
3,client_3ffa76342f366962,content_ab8171c9576be3ba,1.0,1.0,0.0,30,208250,72834,4
4,client_b77d0d5f08f05e64,content_0e3b080033024bc1,2.0,1.0,0.0,3,187832,72870,5
5,client_e00b29e582949543,content_2b95df5a3d475692,2.0,1.0,0.0,8,188694,72883,6
6,client_3ffa76342f366962,content_eaed0db4a9e862d4,2.0,1.0,0.0,30,189695,72901,7
7,client_3ffa76342f366962,content_a559316f945eed2a,1.0,1.0,0.0,30,207983,72918,8
8,client_3ffa76342f366962,content_96b31a38d9a7e56d,1.0,1.0,0.0,30,207191,72658,9
9,client_3ffa76342f366962,content_a477d9dd85e8827f,1.0,1.0,0.0,30,207934,72941,10


In [18]:
# -----------------------------
# ---Compare Top 100 Overlap---
# -----------------------------

top_k = 100

top_impressions = set(
    baseline_comparison.nsmallest(top_k, 'rank_impressions')['content_hash_id']
)

top_clicks = set(
    baseline_comparison.nsmallest(top_k, 'rank_clicks')['content_hash_id']
)

top_position = set(
    baseline_comparison.nsmallest(top_k, 'rank_position')['content_hash_id']
)

overlap_summary = pd.DataFrame({
    'comparison': [
        'Impressions vs Clicks',
        'Impressions vs Position',
        'Clicks vs Position'
    ],
    'overlap_count': [
        len(top_impressions & top_clicks),
        len(top_impressions & top_position),
        len(top_clicks & top_position)
    ]
})

overlap_summary['overlap_percentage'] = (
    overlap_summary['overlap_count'] / top_k * 100
)

overlap_summary

,comparison,overlap_count,overlap_percentage
0,Impressions vs Clicks,30,30.0
1,Impressions vs Position,0,0.0
2,Clicks vs Position,0,0.0


In [19]:
# ----------------------------------
# Compare the Top 100 Characteristic
# ----------------------------------

top_100_summary = pd.DataFrame({
    'baseline': [
        'Impressions',
        'Clicks',
        'Position'
    ],
    'avg_impressions': [
        baseline_comparison.nsmallest(100, 'rank_impressions')['total_gsc_impressions'].mean(),
        baseline_comparison.nsmallest(100, 'rank_clicks')['total_gsc_impressions'].mean(),
        baseline_comparison.nsmallest(100, 'rank_position')['total_gsc_impressions'].mean()
    ],
    'median_impressions': [
        baseline_comparison.nsmallest(100, 'rank_impressions')['total_gsc_impressions'].median(),
        baseline_comparison.nsmallest(100, 'rank_clicks')['total_gsc_impressions'].median(),
        baseline_comparison.nsmallest(100, 'rank_position')['total_gsc_impressions'].median()
    ],
    'avg_clicks': [
        baseline_comparison.nsmallest(100, 'rank_impressions')['total_gsc_clicks'].mean(),
        baseline_comparison.nsmallest(100, 'rank_clicks')['total_gsc_clicks'].mean(),
        baseline_comparison.nsmallest(100, 'rank_position')['total_gsc_clicks'].mean()
    ],
    'avg_position': [
        baseline_comparison.nsmallest(100, 'rank_impressions')['mean_gsc_avg_position'].mean(),
        baseline_comparison.nsmallest(100, 'rank_clicks')['mean_gsc_avg_position'].mean(),
        baseline_comparison.nsmallest(100, 'rank_position')['mean_gsc_avg_position'].mean()
    ]
})

top_100_summary

,baseline,avg_impressions,median_impressions,avg_clicks,avg_position
0,Impressions,141253.52,107293.0,3501.18,7.358882
1,Clicks,83652.71,49769.5,4005.17,4.500519
2,Position,2.46,3.0,0.26,0.000000


In [20]:
# -------------------------
# Correct Position Baseline
# -------------------------

baseline_position = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY mean_gsc_avg_position ASC
        ) AS rank_position

    FROM feature_dataset
    WHERE mean_gsc_avg_position IS NOT NULL
      AND mean_gsc_avg_position > 0
""").df()

baseline_position.head(10)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_position
0,client_1a730cb2640a1abf,content_876803220e3c72dc,6.0,0.0,0.083333,30,1
1,client_23a62021009f63c4,content_9bb345938996f3dc,7.0,0.0,0.100000,30,2
2,client_e00b29e582949543,content_e445b4fe80b72b47,5.0,0.0,0.111111,13,3
3,client_fef1a8f436438636,content_f6cf81886903cb92,6.0,0.0,0.125000,30,4
4,client_cd12bcfd98942aa1,content_04aa38d8ff8ca095,12.0,0.0,0.136364,25,5
5,client_b10cb2997d0c7c86,content_ea8dbea0a2f72bb4,13.0,3.0,0.142857,30,6
6,client_65de48885f4ef01b,content_f38f080cde16b0e5,10.0,0.0,0.142857,30,7
7,client_62f4a7e64f5e0096,content_07105f300b19f668,8.0,0.0,0.166667,30,8
8,client_cd12bcfd98942aa1,content_7339c440160d273d,6.0,0.0,0.166667,25,9
9,client_62f4a7e64f5e0096,content_c5373fea3aeff3d7,6.0,0.0,0.166667,30,10


In [21]:
baseline_comparison = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_impressions DESC
        ) AS rank_impressions,

        ROW_NUMBER() OVER (
            ORDER BY total_gsc_clicks DESC
        ) AS rank_clicks,

        ROW_NUMBER() OVER (
            ORDER BY mean_gsc_avg_position ASC NULLS LAST
        ) AS rank_position

    FROM feature_dataset
    WHERE mean_gsc_avg_position IS NOT NULL
      AND mean_gsc_avg_position > 0
""").df()

baseline_comparison.head(10)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,rank_impressions,rank_clicks,rank_position
0,client_1a730cb2640a1abf,content_876803220e3c72dc,6.0,0.0,0.083333,30,170903,169153,1
1,client_23a62021009f63c4,content_9bb345938996f3dc,7.0,0.0,0.100000,30,167582,165832,2
2,client_e00b29e582949543,content_e445b4fe80b72b47,5.0,0.0,0.111111,13,172372,170622,3
3,client_fef1a8f436438636,content_f6cf81886903cb92,6.0,0.0,0.125000,30,169374,167624,4
4,client_cd12bcfd98942aa1,content_04aa38d8ff8ca095,12.0,0.0,0.136364,25,157037,108224,5
5,client_65de48885f4ef01b,content_f38f080cde16b0e5,10.0,0.0,0.142857,30,160032,111219,6
6,client_b10cb2997d0c7c86,content_ea8dbea0a2f72bb4,13.0,3.0,0.142857,30,154882,36902,7
7,client_62f4a7e64f5e0096,content_07105f300b19f668,8.0,0.0,0.166667,30,163790,114977,8
8,client_cd12bcfd98942aa1,content_7339c440160d273d,6.0,0.0,0.166667,25,169531,167781,9
9,client_62f4a7e64f5e0096,content_c5373fea3aeff3d7,6.0,0.0,0.166667,30,170195,168445,10


## Task 9.9 — Baseline Findings

The baseline comparison shows that simple ranking strategies prioritize substantially different content items.

### Finding 1 — Impressions and clicks are related but not identical

The Top-100 overlap between impressions and clicks rankings was 30%.

This indicates that visibility and actual click activity capture related but different aspects of search performance.

### Finding 2 — Position represents a different performance dimension

The position-based ranking produced little overlap with the impressions and clicks rankings.

This indicates that search visibility, traffic, and average search position should not be treated as interchangeable signals.

### Finding 3 — Position missingness must be handled explicitly

Records with zero or unavailable average position should not automatically be treated as having the best possible position.

A value of zero in this dataset is associated with cases where search impressions are unavailable or zero, so these records are excluded from the valid position ranking.

### Finding 4 — No single simple baseline represents opportunity completely

The impressions baseline favors highly visible content, while the clicks baseline favors content generating more search traffic.

The position baseline favors content with stronger observed search positions.

Because the research objective is to identify content that may deserve review or improvement, a useful opportunity-ranking approach may need to combine multiple signals rather than rely on a single performance metric.

### Baseline Conclusion

The baseline analysis establishes simple reference rankings based on impressions, clicks, and search position.

These baselines provide a transparent reference point for evaluating a later opportunity-scoring approach.

However, they should not be interpreted as ground-truth labels for successful optimization because the dataset does not contain confirmed post-optimization outcomes.

In [ ]:
# -----------------------------
# --Create Normalized Signals--
# -----------------------------

reference_score_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        ctr,
        reporting_days,

        LN(1 + total_gsc_impressions) AS log_impressions

    FROM feature_dataset

    WHERE total_gsc_impressions > 0
      AND mean_gsc_avg_position IS NOT NULL
      AND mean_gsc_avg_position > 0
""").df()

reference_score_data.head()

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,ctr,reporting_days,log_impressions
0,client_1a8bf67cad4ee525,content_66815be25c2aabad,961.0,8.0,5.371188,0.008325,30,6.869014
1,client_1a8bf67cad4ee525,content_66a558a4c3e9df46,400.0,2.0,2.233805,0.005000,30,5.993961
2,client_1a8bf67cad4ee525,content_66c3f7503109aebd,165.0,0.0,3.417626,0.000000,30,5.111988
3,client_1a8bf67cad4ee525,content_66dadf5510782af7,321.0,0.0,2.019991,0.000000,30,5.774552
4,client_1a8bf67cad4ee525,content_67094ffd228da906,237.0,0.0,54.990292,0.000000,20,5.472271


In [23]:
# ----------------------
# ---Visibility Score---
# ----------------------

reference_score_data['visibility_score'] = (
    np.log1p(reference_score_data['total_gsc_impressions'])
    / np.log1p(reference_score_data['total_gsc_impressions']).max()
)

In [ ]:
# --------------------------
# Position Opportunity Score
# --------------------------

reference_score_data['position_opportunity'] = (
    reference_score_data['mean_gsc_avg_position']
    / reference_score_data['mean_gsc_avg_position'].max()
)

In [ ]:
# ---------------------
# ---CTR Opportunity---
# ---------------------

reference_score_data['ctr_opportunity'] = (
    1 - reference_score_data['ctr']
)

In [26]:
# -------------------------
# ---Combine the Signals---
# -------------------------

reference_score_data['reference_opportunity_score'] = (
    0.40 * reference_score_data['visibility_score']
    + 0.40 * reference_score_data['position_opportunity']
    + 0.20 * reference_score_data['ctr_opportunity']
)

In [27]:
reference_score_data = reference_score_data.sort_values(
    'reference_opportunity_score',
    ascending=False
).reset_index(drop=True)

reference_score_data['reference_rank'] = (
    reference_score_data.index + 1
)

reference_score_data.head(20)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,ctr,reporting_days,log_impressions,visibility_score,position_opportunity,ctr_opportunity,reference_opportunity_score,reference_rank
0,client_23a62021009f63c4,content_2304767c893aa3f3,1.0,0.0,579.000000,0.000000,30,0.693147,0.052001,1.000000,1.000000,0.620801,1
1,client_e547b89c05043229,content_963de14b1f58978f,615012.0,1676.0,6.397556,0.002725,30,13.329399,1.000000,0.011049,0.997275,0.603875,2
2,client_23a62021009f63c4,content_c60628276389acbb,123819.0,6.0,73.408960,0.000048,30,11.726584,0.879753,0.126786,0.999952,0.602606,3
3,client_e547b89c05043229,content_eadb33b5df496f4a,591696.0,3817.0,2.258612,0.006451,30,13.290750,0.997100,0.003901,0.993549,0.599110,4
4,client_e547b89c05043229,content_545bb6cc7081ded3,585712.0,3048.0,2.110776,0.005204,30,13.280585,0.996338,0.003646,0.994796,0.598953,5
5,client_8ddc46da5414ffd8,content_943dc881428182b8,292416.0,399.0,2.689206,0.001364,30,12.585936,0.944224,0.004645,0.998636,0.579274,6
6,client_06d356715a8ff3b6,content_f88878f155e4838d,277353.0,2543.0,5.034392,0.009169,30,12.533050,0.940256,0.008695,0.990831,0.577747,7
7,client_e547b89c05043229,content_9ef3d7516483e665,269943.0,787.0,2.098715,0.002915,30,12.505970,0.938225,0.003625,0.997085,0.576157,8
8,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,235427.0,45.0,7.132457,0.000191,30,12.369160,0.927961,0.012319,0.999809,0.576074,9
9,client_a80fca3f171ed1de,content_012de75c008aa653,179662.0,0.0,18.276568,0.000000,30,12.098838,0.907681,0.031566,1.000000,0.575699,10


In [28]:
# ----------------------------
# Inspect the Reference Ranking
# ----------------------------

reference_score_data[
    [
        'client_hash_id',
        'content_hash_id',
        'total_gsc_impressions',
        'total_gsc_clicks',
        'mean_gsc_avg_position',
        'ctr',
        'visibility_score',
        'position_opportunity',
        'ctr_opportunity',
        'reference_opportunity_score',
        'reference_rank'
    ]
].head(20)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,ctr,visibility_score,position_opportunity,ctr_opportunity,reference_opportunity_score,reference_rank
0,client_23a62021009f63c4,content_2304767c893aa3f3,1.0,0.0,579.000000,0.000000,0.052001,1.000000,1.000000,0.620801,1
1,client_e547b89c05043229,content_963de14b1f58978f,615012.0,1676.0,6.397556,0.002725,1.000000,0.011049,0.997275,0.603875,2
2,client_23a62021009f63c4,content_c60628276389acbb,123819.0,6.0,73.408960,0.000048,0.879753,0.126786,0.999952,0.602606,3
3,client_e547b89c05043229,content_eadb33b5df496f4a,591696.0,3817.0,2.258612,0.006451,0.997100,0.003901,0.993549,0.599110,4
4,client_e547b89c05043229,content_545bb6cc7081ded3,585712.0,3048.0,2.110776,0.005204,0.996338,0.003646,0.994796,0.598953,5
5,client_8ddc46da5414ffd8,content_943dc881428182b8,292416.0,399.0,2.689206,0.001364,0.944224,0.004645,0.998636,0.579274,6
6,client_06d356715a8ff3b6,content_f88878f155e4838d,277353.0,2543.0,5.034392,0.009169,0.940256,0.008695,0.990831,0.577747,7
7,client_e547b89c05043229,content_9ef3d7516483e665,269943.0,787.0,2.098715,0.002915,0.938225,0.003625,0.997085,0.576157,8
8,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,235427.0,45.0,7.132457,0.000191,0.927961,0.012319,0.999809,0.576074,9
9,client_a80fca3f171ed1de,content_012de75c008aa653,179662.0,0.0,18.276568,0.000000,0.907681,0.031566,1.000000,0.575699,10


In [29]:
reference_score_data['reference_opportunity_score'].describe()

count    201853.000000
mean          0.352852
std           0.068215
min           0.021491
25%           0.304134
50%           0.358505
75%           0.402713
max           0.620801
Name: reference_opportunity_score, dtype: float64

### Reference Score Revision

The initial reference score showed that extremely low-exposure content could receive a high opportunity score because poor average position had a large influence.

For example, content with only one impression and a very poor average position could rank above content with substantially greater search exposure.

To reduce this effect, the revised reference score:

- Requires a minimum of 100 impressions.
- Uses log-transformed impressions to reduce the influence of extreme exposure values.
- Uses position as an opportunity signal.
- Uses CTR as a supporting opportunity signal.
- Is treated as an analytical benchmark rather than a ground-truth outcome.

In [30]:
# --------------------------
# Create the revised dataset
# --------------------------

reference_score_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        ctr,
        reporting_days,

        LN(1 + total_gsc_impressions) AS log_impressions

    FROM feature_dataset

    WHERE total_gsc_impressions >= 100
      AND mean_gsc_avg_position IS NOT NULL
      AND mean_gsc_avg_position > 0
""").df()

reference_score_data.shape

(101893, 8)

In [31]:
# ---------------------
# Calculate the signals
# ---------------------

reference_score_data['visibility_score'] = (
    reference_score_data['log_impressions']
    / reference_score_data['log_impressions'].max()
)

In [32]:
reference_score_data['position_opportunity'] = (
    reference_score_data['mean_gsc_avg_position']
    / (
        reference_score_data['mean_gsc_avg_position'] + 10
    )
)

In [33]:
reference_score_data['ctr_opportunity'] = (
    1 - reference_score_data['ctr']
)

In [34]:
reference_score_data['reference_opportunity_score'] = (
    0.40 * reference_score_data['visibility_score']
    + 0.40 * reference_score_data['position_opportunity']
    + 0.20 * reference_score_data['ctr_opportunity']
)

In [35]:
reference_score_data = reference_score_data.sort_values(
    'reference_opportunity_score',
    ascending=False
).reset_index(drop=True)

reference_score_data['reference_rank'] = (
    reference_score_data.index + 1
)

In [36]:
# --------------------------
# Validate the Revised Score
# --------------------------

reference_score_data[
    [
        'client_hash_id',
        'content_hash_id',
        'total_gsc_impressions',
        'total_gsc_clicks',
        'mean_gsc_avg_position',
        'ctr',
        'visibility_score',
        'position_opportunity',
        'ctr_opportunity',
        'reference_opportunity_score',
        'reference_rank'
    ]
].head(20)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,ctr,visibility_score,position_opportunity,ctr_opportunity,reference_opportunity_score,reference_rank
0,client_23a62021009f63c4,content_c60628276389acbb,123819.0,6.0,73.408960,0.000048,0.879753,0.880109,0.999952,0.903935,1
1,client_e547b89c05043229,content_ba98b9ba325cb149,103160.0,52.0,41.306458,0.000504,0.866059,0.805093,0.999496,0.868360,2
2,client_3197e6291363b4db,content_65b8a4998e633d89,32279.0,0.0,81.708752,0.000000,0.778895,0.890959,1.000000,0.867942,3
3,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,30278.0,2.0,64.618759,0.000066,0.774094,0.865985,0.999934,0.856019,4
4,client_e547b89c05043229,content_9b76202a02885037,46992.0,14.0,48.407159,0.000298,0.807070,0.828788,0.999702,0.854284,5
5,client_fef1a8f436438636,content_0aaa197051f58d6f,31884.0,9.0,56.514726,0.000282,0.777971,0.849657,0.999718,0.850995,6
6,client_3197e6291363b4db,content_3fd3671dd5e2604f,13228.0,0.0,87.659985,0.000000,0.711973,0.897604,1.000000,0.843831,7
7,client_23a62021009f63c4,content_1b829171cdff7636,34981.0,40.0,39.095720,0.001143,0.784926,0.796316,0.998857,0.832268,8
8,client_23a62021009f63c4,content_36e53e9c707674fc,80453.0,213.0,27.351133,0.002648,0.847408,0.732271,0.997352,0.831342,9
9,client_1a730cb2640a1abf,content_d61fc394d10cba41,28364.0,8.0,42.368346,0.000282,0.769195,0.809045,0.999718,0.831240,10


In [38]:
reference_score_data['reference_opportunity_score'].describe()

count    101893.000000
mean          0.617417
std           0.066260
min           0.339640
25%           0.572556
50%           0.620032
75%           0.667123
max           0.903935
Name: reference_opportunity_score, dtype: float64

In [39]:
reference_score_data[
    [
        'total_gsc_impressions',
        'mean_gsc_avg_position',
        'ctr',
        'reference_opportunity_score'
    ]
].corr()

,total_gsc_impressions,mean_gsc_avg_position,ctr,reference_opportunity_score
total_gsc_impressions,1.000000,-0.144781,0.065754,0.118559
mean_gsc_avg_position,-0.144781,1.000000,-0.239260,0.723242
ctr,0.065754,-0.239260,1.000000,-0.240999
reference_opportunity_score,0.118559,0.723242,-0.240999,1.000000


### Final Reference Score Definition

The final reference score is designed to represent relative content-review opportunity rather than predict a future optimization outcome.

The score prioritizes content that combines:

- Meaningful search visibility
- A relatively weak observed average search position
- Low observed CTR as a supporting signal

To improve interpretability, average position is converted into opportunity bands rather than being used as an unrestricted continuous value.

The score is used only as a reference benchmark for ranking evaluation.
It is not treated as a ground-truth label or a prediction of future performance improvement.

In [40]:
reference_score_data['position_opportunity'] = np.select(
    [
        reference_score_data['mean_gsc_avg_position'] <= 10,
        reference_score_data['mean_gsc_avg_position'] <= 20,
        reference_score_data['mean_gsc_avg_position'] <= 50,
        reference_score_data['mean_gsc_avg_position'] > 50
    ],
    [
        0.25,
        0.50,
        0.75,
        1.00
    ],
    default=np.nan
)

In [41]:
reference_score_data['reference_opportunity_score'] = (
    0.50 * reference_score_data['visibility_score']
    + 0.35 * reference_score_data['position_opportunity']
    + 0.15 * reference_score_data['ctr_opportunity']
)

In [43]:
reference_score_data = reference_score_data.sort_values(
    'reference_opportunity_score',
    ascending=False
).reset_index(drop=True)

reference_score_data['reference_rank'] = (
    reference_score_data.index + 1
)

In [44]:
# ----------------------
# ---Final Validation---
# ----------------------

reference_score_data[
    [
        'client_hash_id',
        'content_hash_id',
        'total_gsc_impressions',
        'total_gsc_clicks',
        'mean_gsc_avg_position',
        'ctr',
        'visibility_score',
        'position_opportunity',
        'ctr_opportunity',
        'reference_opportunity_score',
        'reference_rank'
    ]
].head(20)

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,ctr,visibility_score,position_opportunity,ctr_opportunity,reference_opportunity_score,reference_rank
0,client_23a62021009f63c4,content_c60628276389acbb,123819.0,6.0,73.408960,0.000048,0.879753,1.00,0.999952,0.939869,1
1,client_3197e6291363b4db,content_65b8a4998e633d89,32279.0,0.0,81.708752,0.000000,0.778895,1.00,1.000000,0.889448,2
2,client_fef1a8f436438636,content_0aaa197051f58d6f,31884.0,9.0,56.514726,0.000282,0.777971,1.00,0.999718,0.888943,3
3,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,30278.0,2.0,64.618759,0.000066,0.774094,1.00,0.999934,0.887037,4
4,client_23a62021009f63c4,content_da36aaa1d72bdad4,19157.0,27.0,50.773906,0.001409,0.739754,1.00,0.998591,0.869666,5
5,client_23a62021009f63c4,content_ea7474d92d9701c3,15906.0,26.0,55.759653,0.001635,0.725803,1.00,0.998365,0.862656,6
6,client_fef1a8f436438636,content_81707f944166e061,13760.0,6.0,50.298409,0.000436,0.714931,1.00,0.999564,0.857400,7
7,client_3197e6291363b4db,content_3fd3671dd5e2604f,13228.0,0.0,87.659985,0.000000,0.711973,1.00,1.000000,0.855986,8
8,client_23a62021009f63c4,content_ecbc72c4e65c4466,11743.0,15.0,66.025504,0.001277,0.703040,1.00,0.998723,0.851328,9
9,client_e547b89c05043229,content_a788d093ce1ae235,10977.0,6.0,71.136428,0.000547,0.697980,1.00,0.999453,0.848908,10


In [45]:
reference_score_data['reference_opportunity_score'].describe()

count    101893.000000
mean          0.564103
std           0.079519
min           0.390196
25%           0.506115
50%           0.560868
75%           0.618241
max           0.939869
Name: reference_opportunity_score, dtype: float64

In [46]:
reference_score_data.groupby(
    pd.cut(
        reference_score_data['mean_gsc_avg_position'],
        bins=[0, 10, 20, 50, np.inf],
        labels=['Top 10', '11-20', '21-50', '50+']
    )
)['reference_opportunity_score'].agg(
    ['count', 'mean', 'median']
)

C:\Users\anasm\AppData\Local\Temp\ipykernel_21036\2633568844.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  reference_score_data.groupby(


,count,mean,median
mean_gsc_avg_position,,,
Top 10,43702,0.502439,0.499160
11-20,26518,0.561584,0.557468
21-50,24777,0.636256,0.628876
50+,6896,0.705327,0.699488


## 9.8 Baseline Conclusion

The baseline analysis established several simple and transparent ranking approaches for content opportunity.

The performance-only baselines provide useful reference rankings but capture different aspects of content performance:

- Total impressions prioritize highly visible content.
- Total clicks prioritize content generating the most search clicks.
- Average position provides a visibility-performance reference but is not sufficient as a standalone opportunity ranking because low-exposure items can have extreme position values.

A reference opportunity score was therefore constructed using three observable signals:

- Search visibility through log-transformed impressions
- Position-based opportunity using predefined position bands
- Click-through opportunity using CTR

The reference score was calculated only for content items with at least 100 impressions and a valid positive average position. This minimum exposure threshold reduces the influence of extremely low-exposure items.

The resulting reference score showed a consistent relationship with position bands:

- Top 10: mean score = 0.502
- Positions 11–20: mean score = 0.562
- Positions 21–50: mean score = 0.636
- Positions 50+: mean score = 0.705

This confirms that the reference score behaves consistently with the intended opportunity definition: content with meaningful search exposure but weaker average positions receives higher opportunity scores.

However, this score should not be treated as a ground-truth outcome label. The dataset does not contain a direct post-optimization success indicator. Therefore, the reference score is used as an analytical benchmark for comparing ranking approaches rather than as evidence of future performance improvement.

The baseline phase provides a transparent reference point for evaluating more advanced feature-based or machine learning approaches.